# Part 1

In [ ]:
# Install dependencies
!pip install -q transformers trl peft accelerate bitsandbytes datasets huggingface_hub openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.7 MB/s eta 0:00:00


In [ ]:
# Login
from huggingface_hub import login
login(token = "XXX") # hid my token after use

### I chose OpenAI as the LLM judge

In [ ]:
# Setup OpenAI API key
import os
os.environ["OPENAI_API_KEY"] = "XXX" # hid my API key after use

In [ ]:
# verify GPU
!nvidia-smi

Tue Apr 21 04:44:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### I chose Llama-3.2 1B as the student model

In [ ]:
# Load Llama-3.2 1B model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Meta's official Llama-3.2-1B-Instruct (requires license approval)
model_id = "meta-llama/Llama-3.2-1B-Instruct"
# model_id = "unsloth/Llama-3.2-1B-Instruct"  # fallback if Meta access pending

# 4-bit quantization — reduces memory usage on T4 GPU
# fp16 compute dtype (T4 does not support bf16)
# nf4 + double quant for best memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)
model.config.use_cache = False  # required for DPO training
print("Model successfully loaded!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model successfully loaded!


## Examples

In [ ]:
# Load some prompts from the Alpaca dataset
from datasets import load_dataset

alpaca = load_dataset("tatsu-lab/alpaca", split="train")

# Filter to instruction-only prompts (no input field needed)
prompts = [
    ex["instruction"]
    for ex in alpaca
    if ex["input"].strip() == "" and len(ex["instruction"]) < 200
][:500]

print(f"Loaded {len(prompts)} prompts")
print("Example:", prompts[0])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

Loaded 500 prompts
Example: Give three tips for staying healthy.


In [ ]:
# genearate_response function
def generate_response(prompt, temperature=0.7, max_new_tokens=256):
    messages = [{"role": "user", "content": prompt}]

    # tokenize=False first to get plain string, then tokenize separately
    # this avoids KeyError: 'shape' bug in some transformers versions
    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"]

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(
        output[0][input_ids.shape[-1]:],
        skip_special_tokens=True
    )
    return decoded.strip()

# Quick test
test_prompt = "Explain what machine learning is in simple terms."
resp_a = generate_response(test_prompt, temperature=0.3)
resp_b = generate_response(test_prompt, temperature=1.2)
print("Response A (temp=0.3):", resp_a)
print("---")
print("Response B (temp=1.2):", resp_b)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Response A (temp=0.3): Machine learning is a way for computers to learn from data without being explicitly programmed. It's like teaching a child to recognize pictures of cats and dogs. 

Imagine you have a bunch of pictures of cats and dogs. You want to teach a computer to recognize cats and dogs. 

Here's how it works:

1. **Training**: You show the computer a bunch of pictures of cats and dogs, and tell it what they are.
2. **Learning**: The computer looks at the pictures and tries to figure out what makes a cat and a dog.
3. **Pattern recognition**: The computer starts to recognize patterns in the pictures, like the shape of a cat's ears or the way a dog walks.
4. **Prediction**: When you show the computer a new picture, it uses the patterns it learned to make a prediction about what it thinks the picture is.
5. **Improvement**: The computer improves its predictions over time, so it can recognize more cats and dogs.

Machine learning is used in many things, like:

- Image recogniti

In [ ]:
# Set up OpenAI as the judge, and write the judge function
from openai import OpenAI

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def judge_responses(prompt, response_a, response_b):
    judge_prompt = f"""You are an expert AI assistant evaluator. Given a prompt and two responses, determine which response is better.

Evaluate based on:
- Accuracy and correctness
- Clarity and coherence
- Helpfulness and completeness
- Overall quality

Prompt: {prompt}

Response A:
{response_a}

Response B:
{response_b}

Which response is better? Reply with ONLY 'A' or 'B', nothing else."""

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": judge_prompt}],
        temperature=0.0,  # deterministic judging
        max_tokens=1
    )
    verdict = response.choices[0].message.content.strip().upper()
    return verdict

# Test the judge
verdict = judge_responses(test_prompt, resp_a, resp_b)
print(f"Judge picked: Response {verdict}")

Judge picked: Response A


In [ ]:
# Build the full preference dataset
import time
import json

preference_data = []

for i, prompt in enumerate(prompts[:200]):
    try:
        print(f"[{i+1}/200] Generating responses...", end=" ")
        resp_a = generate_response(prompt, temperature=0.3)
        resp_b = generate_response(prompt, temperature=1.2)

        print("Judging...", end=" ")
        verdict = judge_responses(prompt, resp_a, resp_b)

        if verdict == "A":
            chosen, rejected = resp_a, resp_b
        else:
            chosen, rejected = resp_b, resp_a

        preference_data.append({
            "prompt": prompt,
            "chosen": chosen,
            "rejected": rejected
        })

        print(f"✓ Winner: {verdict} | Total: {len(preference_data)}")

        time.sleep(0.5)  # OpenAI has generous rate limits, 0.5s is enough

    except Exception as e:
        print(f"✗ Skipping prompt {i} due to error: {e}")
        continue

print(f"\nDone! Dataset size: {len(preference_data)} examples")

with open("preference_dataset.json", "w") as f:
    json.dump(preference_data, f, indent=2)
print("Saved!")

[1/200] Generating responses... Judging... ✓ Winner: A | Total: 1
[2/200] Generating responses... Judging... ✓ Winner: B | Total: 2
[3/200] Generating responses... Judging... ✓ Winner: A | Total: 3
[4/200] Generating responses... Judging... ✓ Winner: B | Total: 4
[5/200] Generating responses... Judging... ✓ Winner: B | Total: 5
[6/200] Generating responses... Judging... ✓ Winner: A | Total: 6
[7/200] Generating responses... Judging... ✓ Winner: B | Total: 7
[8/200] Generating responses... Judging... ✓ Winner: A | Total: 8
[9/200] Generating responses... Judging... ✓ Winner: A | Total: 9
[10/200] Generating responses... Judging... ✓ Winner: B | Total: 10
[11/200] Generating responses... Judging... ✓ Winner: A | Total: 11
[12/200] Generating responses... Judging... ✓ Winner: A | Total: 12
[13/200] Generating responses... Judging... ✓ Winner: A | Total: 13
[14/200] Generating responses... Judging... ✓ Winner: A | Total: 14
[15/200] Generating responses... Judging... ✓ Winner: B | Total: 1

## Document your reasoning for the judge's prompt design

### The judge prompt was designed with four evaluation criteria: accuracy, clarity, helpfulness, and completeness, to provide a comprehensive assessment of response quality. The output was constrained to a single token ('A' or 'B') to eliminate ambiguity and ensure deterministic, parseable responses that can be directly used as training labels. Temperature was set to 0.0 to make judgments reproducible across runs. Llama-3.3-70B via Groq was chosen as the judge because it is significantly more capable than the student model Llama-1B, ensuring reliable quality distinctions.

## Explain how you ensure consistent and reliable preference judgments

### Consistency and reliability were ensured through three mechanisms. First, temperature was set to 0.0, making the judge deterministic so identical inputs always produce the same verdict. Second, constraining output to a single token forces a binary choice, eliminating variability from verbose reasoning. Third, using Llama-3.3-70B as the judge, a model significantly more capable than the student model Llama-1B, ensures the judge has sufficient understanding to distinguish response quality reliably. Together these design choices minimize noise in the preference labels, which is critical for stable DPO training.

# Part 2

## DPO Fine-tuning

In [ ]:
# Convert dataset to HF
from datasets import Dataset

def format_example(example):
    # Format using Llama's exact special tokens for consistent tokenization
    prompt_msg = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{example['prompt']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
    return {
        "prompt": prompt_msg,
        "chosen": example["chosen"] + "<|eot_id|>",
        "rejected": example["rejected"] + "<|eot_id|>"
    }

formatted_data = [format_example(ex) for ex in preference_data]
hf_dataset = Dataset.from_list(formatted_data)
hf_dataset = hf_dataset.train_test_split(test_size=0.1, seed=42)
print(hf_dataset)

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 180
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 20
    })
})


In [ ]:
# Setup LoRA config
from peft import LoraConfig, TaskType

# LoRA trains a small adapter instead of the full model
# r=16: rank controls adapter capacity
# lora_alpha=32: scaling factor
# target_modules: attention layers to apply LoRA to
# inference_mode=False: required for training
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False
)

print("LoRA config ready")

LoRA config ready


In [ ]:
# DPO training
from trl import DPOTrainer, DPOConfig

# DPO trains model to prefer chosen over rejected responses
# beta=0.1: controls how strongly to enforce preferences
# fp16/bf16 both False: let 4-bit quantization handle precision (T4 limitation)
dpo_config = DPOConfig(
    output_dir="./dpo-llama-1b",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    beta=0.1,
    logging_steps=10,
    save_steps=50,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    bf16=False,
    fp16=False,
    remove_unused_columns=False,
    report_to="none",
    max_length=512,
)

trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=hf_dataset["train"],
    eval_dataset=hf_dataset["test"],
    processing_class=tokenizer,
    peft_config=lora_config,
)

print("Starting DPO training...")
trainer.train()
print("Training complete!")

Adding EOS to train dataset:   0%|          | 0/180 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/180 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Starting DPO training...


Step,Training Loss
10,0.691040
20,0.679041
30,0.553156
40,0.508598


Training complete!


In [ ]:
# Save to HF

# Save LoRA adapter locally (only ~7MB vs 2.47GB full model)
trainer.model.save_pretrained("./dpo-llama-1b-adapter")
tokenizer.save_pretrained("./dpo-llama-1b-adapter")
print("Saved locally!")

# Push to HuggingFace
repo_name = "alanshi31/dpo-llama-1b-openai-judge"

trainer.model.push_to_hub(repo_name, private=False)
tokenizer.push_to_hub(repo_name, private=False)
print(f"Done! View at: https://huggingface.co/{repo_name}")

Saved locally!


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   9%|9         |  622kB / 6.83MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpg3auw7ig/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

Done! View at: https://huggingface.co/alanshi31/dpo-llama-1b-openai-judge


*Repo link above

## Comparative Analysis

In [ ]:
# Load both models
from peft import PeftModel
import pandas as pd

# Load base model fresh for fair comparison
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

# DPO model = base model + trained LoRA adapter
dpo_model = PeftModel.from_pretrained(base_model, "./dpo-llama-1b-adapter")
print("Both models loaded!")

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Both models loaded!


### 10 novel instructions

In [ ]:
# Generate Analysis DataFrame
# 10 novel prompts not seen during training
novel_prompts = [
    "Explain the difference between supervised and unsupervised learning.",
    "What are three strategies to improve focus while studying?",
    "Describe how the internet works in simple terms.",
    "What is the difference between a virus and a bacteria?",
    "Give advice on how to negotiate a higher salary.",
    "Explain why sleep is important for health.",
    "What are the pros and cons of electric vehicles?",
    "How does compound interest work? Give an example.",
    "What is the difference between a CPU and a GPU for deep learning?",
    "How would you explain neural networks to a 10 year old?"
]

def generate_with_model(mdl, prompt, temperature=0.7, max_new_tokens=200):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(mdl.device)
    input_ids = inputs["input_ids"]
    with torch.no_grad():
        output = mdl.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(
        output[0][input_ids.shape[-1]:], skip_special_tokens=True
    ).strip()

results = []
for i, prompt in enumerate(novel_prompts):
    print(f"Generating {i+1}/10...")
    base_resp = generate_with_model(base_model, prompt)
    dpo_resp = generate_with_model(dpo_model, prompt)
    results.append({
        "Prompt": prompt,
        "Base Llama-1B": base_resp,
        "DPO Fine-tuned": dpo_resp
    })

df = pd.DataFrame(results)
df.to_csv("comparative_analysis.csv", index=False)
print("\nDone!")
df

Generating 1/10...
Generating 2/10...
Generating 3/10...
Generating 4/10...
Generating 5/10...
Generating 6/10...
Generating 7/10...
Generating 8/10...
Generating 9/10...
Generating 10/10...

Done!


,Prompt,Base Llama-1B,DPO Fine-tuned
0,Explain the difference between supervised and ...,Supervised and unsupervised learning are two t...,Supervised and unsupervised learning are two f...
1,What are three strategies to improve focus whi...,Here are three strategies to improve focus whi...,Here are three strategies to improve focus whi...
2,Describe how the internet works in simple terms.,Here's a simple explanation of how the interne...,The internet is a network of computers that co...
3,What is the difference between a virus and a b...,Viruses and bacteria are two types of microorg...,Viruses and bacteria are two distinct types of...
4,Give advice on how to negotiate a higher salary.,Negotiating a higher salary can be a challengi...,Negotiating a higher salary can be a challengi...
5,Explain why sleep is important for health.,Sleep is essential for overall health and well...,Sleep is one of the most essential aspects of ...
6,What are the pros and cons of electric vehicles?,Electric vehicles (EVs) have gained significan...,Electric vehicles (EVs) have become increasing...
7,How does compound interest work? Give an example.,Compound interest is the interest calculated o...,Compound interest is a powerful financial conc...
8,What is the difference between a CPU and a GPU...,"In deep learning, the main difference between ...","In deep learning, a Central Processing Unit (C..."
9,How would you explain neural networks to a 10 ...,"Imagine you have a big box of cookies, and you...",Imagine you have a big library with millions o...


### Completions

In [ ]:
# Quantitative comparison of base vs DPO model responses
avg_base_len = df["Base Llama-1B"].apply(len).mean()
avg_dpo_len = df["DPO Fine-tuned"].apply(len).mean()
diff = avg_dpo_len - avg_base_len
pct = (diff / avg_base_len) * 100

print(f"Average response length:")
print(f"  Base Llama-1B:  {avg_base_len:.0f} characters")
print(f"  DPO Fine-tuned: {avg_dpo_len:.0f} characters")
print(f"  Difference:     {diff:.0f} characters ({pct:.1f}%)")

# Count how many DPO responses are longer than base
dpo_longer = sum(
    len(row["DPO Fine-tuned"]) > len(row["Base Llama-1B"])
    for _, row in df.iterrows()
)
print(f"\nDPO response longer than base: {dpo_longer}/10 prompts")

Average response length:
  Base Llama-1B:  954 characters
  DPO Fine-tuned: 961 characters
  Difference:     7 characters (0.7%)

DPO response longer than base: 5/10 prompts


In [ ]:
# Print full response for analysis
for i, row in df.iterrows():
    print(f"\n{'='*60}")
    print(f"PROMPT {i+1}: {row['Prompt']}")
    print(f"\n--- Base Llama-1B ---")
    print(row['Base Llama-1B'])
    print(f"\n--- DPO Fine-tuned ---")
    print(row['DPO Fine-tuned'])


PROMPT 1: Explain the difference between supervised and unsupervised learning.

--- Base Llama-1B ---
Supervised and unsupervised learning are two types of machine learning techniques used in artificial intelligence (AI) to train computers to recognize patterns in data. The main difference between them lies in the type of data they process and the goals they aim to achieve.

**Supervised Learning:**

In supervised learning, the algorithm is trained on a labeled dataset, where the goal is to learn a mapping between input data and their corresponding outputs. The algorithm is trained to learn a specific relationship between the input and output variables.

Here's a step-by-step explanation:

1. **Labeled data**: The algorithm is fed a labeled dataset, where each sample is paired with a corresponding output.
2. **Training**: The algorithm learns a mapping between the input features and the target variable.
3. **Prediction**: Once trained, the algorithm can make predictions on new, unseen

# Observation

In this experiment, we fine-tuned Llama-3.2-1B-Instruct using Direct Preference Optimization (DPO) with a preference dataset of 200 examples generated via an LLM-as-a-Judge pipeline, using GPT-4o-mini as the referee model.

__Qualitative Differences:__
Both models produced structured, coherent responses with markdown formatting. The DPO model showed marginally improved technical precision in several prompts. In Prompt 4 (virus vs bacteria), the DPO model correctly referenced peptidoglycan cell walls in its structural description, while the base model only mentioned a generic "cell wall or membrane." In Prompt 3 (how the internet works), the DPO model introduced a more organized taxonomy covering ISPs, DNS, and web browsers, while the base model gave a more narrative step-by-step explanation. In Prompt 10 (neural networks for a 10 year old), both models used analogies but the base model used cookies while the DPO model used a library — neither is objectively better, illustrating that DPO does not always produce a clearly superior response.

__Training Stability:__
Training loss decreased steadily from 0.691 at step 10 to 0.509 at step 40 across 2 epochs on 180 training examples, indicating stable convergence without signs of mode collapse or reward hacking, which are common failure modes in preference optimization.

__Computational Efficiency:__
Training completed in approximately 3 minutes on a T4 GPU using 4-bit quantization and LoRA adapters. The resulting adapter was only 6.83MB compared to the 2.47GB base model, demonstrating that DPO is highly resource-efficient. The full pipeline including dataset collection, training, and evaluation ran within a single Colab session.

__Limitations and Failure Modes:__
Quantitatively, the DPO model averaged 961 characters per response compared to 954 for the base model, a negligible 0.7% difference, with DPO responses longer in only 5 out of 10 prompts. Both models exhibited response truncation due to the 200 token limit, cutting off answers mid-sentence in several prompts. These results suggest that 200 preference pairs and 2 training epochs are insufficient to produce dramatic behavioral changes in a 1B parameter model.

__Suggestions for Improvement:__
Future improvements could include collecting 500+ preference pairs with more diverse prompt categories to reduce distribution shift between training and evaluation. Experimenting with higher beta values in DPO could enforce stronger preference boundaries, potentially producing more pronounced behavioral differences. Additionally, implementing iterative DPO as described in Self-Rewarding Language Models, where the model acts as its own judge across multiple training rounds, could progressively improve alignment without relying on external APIs.